# RuleFit: aturan "JIKA … DAN … MAKA" yang bisa dibaca juri, dari data teks + tabular
**Kapan dipakai:** presentasi butuh penjelasan yang lebih mudah dari SHAP, misalnya
"JIKA ulasan memuat 'return' DAN umur > 40 MAKA peluang direkomendasikan turun". Skornya
biasanya di antara regresi logistik dan GBDT, jadi dipakai untuk **penjelasan + cek akal sehat**,
bukan sebagai model utama.

Cara kerja (Friedman & Popescu 2008):
1. Latih pohon dangkal (gradient boosting, kedalaman `TREE_DEPTH`).
2. Setiap simpul pohon = satu aturan (gabungan syarat dari akar ke simpul). `decision_path`
   memberi matriks 0/1 "baris memenuhi aturan".
3. Regresi logistik L1 di atas [aturan + fitur linear yang dipotong ekor & distandarkan]. L1
   membuat sebagian besar aturan berbobot 0, jadi yang tersisa sedikit dan bisa dibaca.

Fitur: angka, kategori one-hot (`kolom=nilai`), statistik teks, dan kata kunci biner
(`kata:xxx`, dipilih dengan chi² di train).

In [ ]:
import os

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, hstack
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_selection import chi2
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

## Konfigurasi

In [ ]:
OUTPUT_FOLDER = "../../../outputs/text/rulefit_aturan"
DATA_FOLDER = "../../../data_latihan/text/teks_tabular"      # GANTI saat lomba
TRAIN_CSV = f"{DATA_FOLDER}/baju_wanita_train.csv"           # GANTI saat lomba
TEST_CSV = f"{DATA_FOLDER}/baju_wanita_test.csv"             # GANTI saat lomba
TARGET = "recommended_ind"                   # GANTI saat lomba (biner, `rating` bocor)
POSITIVE = 1                                 # GANTI: kelas dijelaskan (multikelas: 1 vs sisa)
NUM_COLS = ["age", "positive_feedback_count"]                    # GANTI saat lomba
CAT_COLS = ["division_name", "department_name", "class_name"]    # GANTI saat lomba
TEXT_COLS = ["title", "review_text"]                             # GANTI saat lomba
STOP_WORDS = "english"       # GANTI: None atau daftar stopword Indonesia
N_WORDS = 40
N_TREES = 40
TREE_DEPTH = 3
RULE_C = 0.05                # makin kecil = aturan makin sedikit
N_FOLDS = 5
SEED = 42


def metric(truth, proba):
    return roc_auc_score(truth, proba)       # GANTI: metrik lomba (proba = peluang POSITIVE)


print("uji metrik:", metric(np.array([0, 1, 1]), np.array([0.2, 0.7, 0.9])))

## Baca data + pisahkan kunci demo
Data demo menyimpan target di file test. Baris "kunci demo" HAPUS saat lomba.

In [ ]:
train = pd.read_csv(TRAIN_CSV)
test = pd.read_csv(TEST_CSV)
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
truth_test = (test[TARGET] == POSITIVE).to_numpy(int)          # kunci demo (HAPUS saat lomba)
test = test.drop(columns=TARGET)                               # kunci demo (HAPUS saat lomba)
y = (train[TARGET] == POSITIVE).to_numpy(int)
both = pd.concat([train, test], ignore_index=True)
print("train:", train.shape, "| test:", test.shape, "| porsi positif:", round(y.mean(), 3))

## Fitur yang mudah dibaca

In [ ]:
features = pd.DataFrame(index=both.index)
for col in NUM_COLS:
    features[col] = both[col]
features = pd.concat([features, pd.get_dummies(both[CAT_COLS], prefix_sep="=", dtype=float)],
                     axis=1)
full_text = both[TEXT_COLS].fillna("").agg(" ".join, axis=1)
features["panjang_kata"] = full_text.str.split().str.len()
for col in TEXT_COLS:
    features[f"{col}_kosong"] = both[col].isna().astype(float)
words = CountVectorizer(binary=True, min_df=50, max_df=0.5, stop_words=STOP_WORDS)
word_matrix = words.fit_transform(full_text)
scores, _ = chi2(word_matrix[: len(train)], y)
top_words = np.argsort(-np.nan_to_num(scores))[:N_WORDS]
vocab = words.get_feature_names_out()
for number in top_words:
    features[f"kata:{vocab[number]}"] = word_matrix[:, number].toarray().ravel()
FEATURE_NAMES = list(features.columns)
BINARY = set(FEATURE_NAMES) - set(NUM_COLS) - {"panjang_kata"}
values_all = features.to_numpy(float)
values_train = values_all[: len(train)]
values_test = values_all[len(train):]
print("fitur:", values_all.shape, "| contoh kata:", list(vocab[top_words[:10]]))

## Aturan dari pohon: teks syarat + matriks 0/1

In [ ]:
SIGNS = {True: "<=", False: ">"}


def parent_map(tree):
    parents = {}
    for node in range(tree.node_count):
        if tree.children_left[node] != -1:
            parents[tree.children_left[node]] = (node, True)
            parents[tree.children_right[node]] = (node, False)
    return parents


def condition_text(name, went_left, threshold):
    if name in BINARY:
        if went_left:
            return f"TIDAK {name}"
        return name
    return f"{name} {SIGNS[went_left]} {threshold:.3g}"


print("uji syarat:", condition_text("age", False, 40.5), condition_text("kata:x", True, 0.5))

In [ ]:
def describe_node(tree, node, parents):
    parts = []
    while node in parents:
        parent, went_left = parents[node]
        name = FEATURE_NAMES[tree.feature[parent]]
        parts.append(condition_text(name, went_left, tree.threshold[parent]))
        node = parent
    return " DAN ".join(parts[::-1])


print("fungsi uraian siap")

In [ ]:
def rule_names(booster):
    names = []
    for estimator in booster.estimators_[:, 0]:
        tree = estimator.tree_
        parents = parent_map(tree)
        for node in range(1, tree.node_count):
            names.append(describe_node(tree, node, parents))
    return names


def rule_matrix(booster, values):
    blocks = []
    for estimator in booster.estimators_[:, 0]:
        blocks.append(estimator.decision_path(values)[:, 1:])
    return hstack(blocks).tocsr()


print("fungsi aturan siap")

## Latih RuleFit + prediksi
Fitur linear: ekor dipotong di persentil 2,5/97,5 lalu distandarkan × 0,4 (saran makalah) agar
setara dengan aturan 0/1.

In [ ]:
def rulefit_fit(values, labels):
    booster = GradientBoostingClassifier(n_estimators=N_TREES, max_depth=TREE_DEPTH,
                                         learning_rate=0.1, subsample=0.5, random_state=SEED)
    booster.fit(values, labels)
    low = np.percentile(values, 2.5, axis=0)
    high = np.percentile(values, 97.5, axis=0)
    clipped = np.clip(values, low, high)
    scale = (clipped.mean(axis=0), clipped.std(axis=0) + 1e-9)
    fitted = (booster, low, high, scale, None)
    model = LogisticRegression(penalty="l1", C=RULE_C, solver="liblinear")
    model.fit(rulefit_inputs(fitted, values), labels)
    return (booster, low, high, scale, model)


def rulefit_inputs(fitted, values):
    booster, low, high, scale, _ = fitted
    linear = 0.4 * (np.clip(values, low, high) - scale[0]) / scale[1]
    return hstack([rule_matrix(booster, values), csr_matrix(linear)]).tocsr()


def rulefit_proba(fitted, values):
    return fitted[4].predict_proba(rulefit_inputs(fitted, values))[:, 1]


print("fungsi RuleFit siap")

## Banding CV: L1 linear vs RuleFit vs LightGBM (lipatan sama)

In [ ]:
def linear_only(values_fit, labels, values_val):
    model = LogisticRegression(penalty="l1", C=RULE_C, solver="liblinear", max_iter=2000)
    mean, std = values_fit.mean(axis=0), values_fit.std(axis=0) + 1e-9
    model.fit((values_fit - mean) / std, labels)
    return model.predict_proba((values_val - mean) / std)[:, 1]


def rulefit_only(values_fit, labels, values_val):
    return rulefit_proba(rulefit_fit(values_fit, labels), values_val)


def lgbm_only(values_fit, labels, values_val):
    model = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, verbose=-1,
                               random_state=SEED).fit(values_fit, labels)
    return model.predict_proba(values_val)[:, 1]


folds = list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED).split(values_train, y))
rows = []
RUNNERS = {"L1 linear": linear_only, "RuleFit": rulefit_only, "LightGBM": lgbm_only}
for name, runner in RUNNERS.items():
    oof = np.zeros(len(y))
    for idx_fit, idx_val in folds:
        oof[idx_val] = runner(values_train[idx_fit], y[idx_fit], values_train[idx_val])
    rows.append({"model": name, "CV": round(metric(y, oof), 4)})
    print(rows[-1])
pd.DataFrame(rows).to_csv(f"{OUTPUT_FOLDER}/banding_model.csv", index=False)

## Aturan terpenting (model di seluruh train)
Penting = |koef| × √(s(1 − s)), s = porsi baris yang memenuhi aturan. Koef positif = menaikkan
peluang kelas `POSITIVE`. Aturan kembar dari pohon berbeda digabung (bobotnya dijumlah).

In [ ]:
FINAL = rulefit_fit(values_train, y)
support = np.asarray(rule_matrix(FINAL[0], values_train).mean(axis=0)).ravel()
names = rule_names(FINAL[0])
coefs = FINAL[4].coef_[0]
rules = pd.DataFrame({"aturan": names, "koef": coefs[: len(names)], "porsi baris": support})
rules = rules[rules["koef"] != 0]
rules = rules.groupby("aturan", as_index=False).agg({"koef": "sum", "porsi baris": "first"})
spread = np.sqrt(rules["porsi baris"] * (1 - rules["porsi baris"]))
rules["penting"] = (rules["koef"].abs() * spread).round(4)
rules = rules.sort_values("penting", ascending=False)
linear = pd.DataFrame({"aturan": FEATURE_NAMES, "koef": coefs[len(names):]})
linear = linear[linear["koef"] != 0]
rules.to_csv(f"{OUTPUT_FOLDER}/aturan_teratas.csv", index=False)
linear.to_csv(f"{OUTPUT_FOLDER}/fitur_linear.csv", index=False)
print("aturan dipakai:", len(rules), "dari", len(names), "| fitur linear dipakai:", len(linear))
print(rules.head(15).to_string(index=False))

## Gambar aturan (bahan slide) + cek kunci demo

In [ ]:
top = rules.head(12).iloc[::-1]
figure, ax = plt.subplots(figsize=(11, 5.5))
colors = np.where(top["koef"] > 0, "#0072B2", "#D55E00")
ax.barh(top["aturan"].str.slice(0, 70), top["penting"], color=colors)
ax.set_xlabel("penting (biru = menaikkan peluang, oranye = menurunkan)")
ax.tick_params(axis="y", labelsize=8)
figure.savefig(f"{OUTPUT_FOLDER}/aturan_teratas.png", dpi=200, bbox_inches="tight")
plt.show()
test_proba = rulefit_proba(FINAL, values_test)
pd.DataFrame({"peluang": test_proba}).to_csv(f"{OUTPUT_FOLDER}/peluang_test.csv", index=False)
print("skor test RuleFit (kunci demo):", round(metric(truth_test, test_proba), 4))